In [ ]:

import os
import telebot
import threading
import time
import re
import base64
import io
from openai import OpenAI

# توکن ربات تلگرام
BOT_TOKEN = "8162451742:AAFb3N0F54HzRgm4vNszQGMSel5fRM57VW0"
bot = telebot.TeleBot(BOT_TOKEN)

# تنظیم کلاینت برای اتصال به API
api_client = OpenAI(
    base_url="https://api.aimlapi.com/v1",
    api_key="api"
)

# vision-enabled 
VISION_MODEL = "gpt-4o-mini"

@bot.message_handler(commands=['start', 'help'])
def welcome_message(message):
    bot.reply_to(message, "سلام! من Aiocoder از ایولرنم. هر سوالی در مورد برنامه‌نویسی داری، بگو تا کمکت کنم! عکس هم می‌تونی بفرستی برای تحلیل.")


def display_waiting_message(bot, chat_id, status):
    time.sleep(5)
    if not status["done"]:
        status["msg"] = bot.send_message(chat_id, "یه لحظه صبر کن، دارم پاسخ رو آماده می‌کنم...")

# جدا کردن دقیق متن اولیه، کد و متن نهایی
def parse_response(response_text):
    # ابتدا چک کن که آیا بلاک کد وجود داره
    code_match = re.search(r"```(?:\w*\n)?([\s\S]+?)```", response_text)
    if code_match:
        code = code_match.group(1).strip()
        # متن اولیه (قبل از کد)
        initial_text = response_text[:code_match.start()].strip()
        # متن نهایی (بعد از کد)
        final_text = response_text[code_match.end():].strip()
        return initial_text, code, final_text
    else:
        # اگه بلاک کد نبود، ممکنه کد به‌صورت ساده باشه (مثلاً با indent)
        code_pattern = re.compile(r"^\s*[\w\.]+\s*\(.*\)\s*:[\s\S]+?(?=\n\n|\Z)", re.MULTILINE)
        code_match = code_pattern.search(response_text)
        if code_match:
            code = code_match.group(0).strip()
            initial_text = response_text[:code_match.start()].strip()
            final_text = response_text[code_match.end():].strip()
            return initial_text, code, final_text
        # اگه هیچ کدی نبود، همه رو به‌عنوان متن اولیه حساب کن
        return response_text.strip(), None, None

# پردازش پیام‌های متنی کاربر
@bot.message_handler(func=lambda msg: True)
def handle_user_message(message):
    user_input = message.text
    chat_id = message.chat.id
    status = {"done": False, "msg": None}

    # اجرای پیام انتظار در یه نخ جدا
    waiting_thread = threading.Thread(target=display_waiting_message, args=(bot, chat_id, status))
    waiting_thread.start()

    # درخواست به API برای گرفتن پاسخ
    response = api_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "تو Aiocoder هستی، برنامه‌نویس حرفه‌ای توی ایولرن. "
                    "با لحن صمیمی و حرفه‌ای به سوالات برنامه‌نویسی جواب بده. "
                    "هر چی کاربر پرسید، کامل و دقیق پاسخ بده و انگار که خودت کد زدی! "
                    "اگه کسی پرسید کی تو رو ساخته، بگو: 'من توسط تیم فنی ایولرن ساخته شدم.' "
                    "هیچ اشاره‌ای به OpenAI، ChatGPT یا هر هوش مصنوعی دیگه نکن. "
                    "پاسخ‌ها رو به سه بخش جداگانه بنویس: "
                    "1. اول یه متن توضیحی (اگه لازم باشه)، "
                    "2. بعدش خود کد با فرمت بلاک کد (```)، "
                    "3. آخرش توضیحات اضافی (اگه باشه)."
                    "اگه کد بلاک کد (```) نداشت، به‌صورت خودکار تشخیصش بده (مثلاً با indent). "
                    "از عباراتی مثل 'توضیح قبل کد' یا 'بعد کد' استفاده نکن، فقط طبیعی بنویس."
                )
            },
            {"role": "user", "content": user_input}
        ],
        stream=True
    )

    # جمع‌آوری پاسخ از استریم
    full_response = ""
    for chunk in response:
        if chunk.choices and chunk.choices[0].delta.content:
            full_response += chunk.choices[0].delta.content

    status["done"] = True

    # حذف پیام انتظار اگه وجود داشت
    if status.get("msg"):
        try:
            bot.delete_message(chat_id, status["msg"].message_id)
        except Exception:
            pass

    # جدا کردن متن اولیه، کد و متن نهایی
    initial_text, code, final_text = parse_response(full_response)

    # ارسال متن اولیه (اگه وجود داشت)
    if initial_text:
        bot.reply_to(message, initial_text)

    # ارسال کد به صورت بلاک (اگه وجود داشت)
    if code:
        bot.send_message(chat_id, f"```\n{code}\n```", parse_mode="Markdown")

    # ارسال متن نهایی (اگه وجود داشت)
    if final_text:
        bot.send_message(chat_id, final_text)

    # اگه هیچی نبود، پیام خطا
    if not initial_text and not code and not final_text:
        bot.reply_to(message, "اوپس! چیزی گیرم نیومد. دوباره بپرس تا درستش کنم!")

# پردازش عکس‌های کاربر (برای تحلیل ارور کد و غیره)
@bot.message_handler(content_types=['photo'])
def handle_photo_message(message):
    chat_id = message.chat.id
    status = {"done": False, "msg": None}

    # اجرای پیام انتظار
    waiting_thread = threading.Thread(target=display_waiting_message, args=(bot, chat_id, status))
    waiting_thread.start()

    try:
        # دانلود عکس
        file_info = bot.get_file(message.photo[-1].file_id)
        downloaded_file = bot.download_file(file_info.file_path)

        # تبدیل به base64
        img_bytes = io.BytesIO(downloaded_file)
        img_base64 = base64.b64encode(img_bytes.getvalue()).decode('utf-8')

        # پیام کاربر (caption اگه باشه)
        caption = message.caption or "این عکس رو تحلیل کن، احتمالاً ارور کد یا چیزی شبیهشه. راه‌حل بده."

        # درخواست به API با vision
        response = api_client.chat.completions.create(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "تو Aiocoder هستی، برنامه‌نویس حرفه‌ای توی ایولرن. "
                        "عکس‌ها رو تحلیل کن، مخصوصاً ارورهای کد یا اسکرین‌شات‌های برنامه‌نویسی. "
                        "با لحن صمیمی و حرفه‌ای جواب بده. "
                        "اگه کسی پرسید کی تو رو ساخته، بگو: 'من توسط تیم فنی ایولرن ساخته شدم.' "
                        "هیچ اشاره‌ای به OpenAI، ChatGPT یا هر هوش مصنوعی دیگه نکن. "
                        "پاسخ‌ها رو به سه بخش جداگانه بنویس: "
                        "1. اول یه متن توضیحی (اگه لازم باشه)، "
                        "2. بعدش خود کد با فرمت بلاک کد (```)، "
                        "3. آخرش توضیحات اضافی (اگه باشه)."
                        "اگه کد بلاک کد (```) نداشت، به‌صورت خودکار تشخیصش بده (مثلاً با indent). "
                        "از عباراتی مثل 'توضیح قبل کد' یا 'بعد کد' استفاده نکن، فقط طبیعی بنویس."
                    )
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": caption},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{img_base64}"
                            }
                        }
                    ]
                }
            ],
            stream=True
        )

        # جمع‌آوری پاسخ از استریم
        full_response = ""
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                full_response += chunk.choices[0].delta.content

        status["done"] = True

        # حذف پیام انتظار
        if status.get("msg"):
            try:
                bot.delete_message(chat_id, status["msg"].message_id)
            except Exception:
                pass

        # جدا کردن متن اولیه، کد و متن نهایی
        initial_text, code, final_text = parse_response(full_response)

        # ارسال متن اولیه
        if initial_text:
            bot.send_message(chat_id, initial_text)

        # ارسال کد به صورت بلاک
        if code:
            bot.send_message(chat_id, f"```\n{code}\n```", parse_mode="Markdown")

        # ارسال متن نهایی
        if final_text:
            bot.send_message(chat_id, final_text)

        # اگه هیچی نبود، پیام خطا
        if not initial_text and not code and not final_text:
            bot.send_message(chat_id, "اوپس! نتونستم عکس رو خوب تحلیل کنم. دوباره امتحان کن یا توضیح بیشتری بده!")

    except Exception as e:
        status["done"] = True
        if status.get("msg"):
            try:
                bot.delete_message(chat_id, status["msg"].message_id)
            except Exception:
                pass
        bot.send_message(chat_id, "یه مشکلی پیش اومد با عکس. مطمئن شو که عکس واضحه و دوباره بفرست!")

# استارت ربات
if __name__ == "__main__":
    print("ربات شروع به کار کرد...")
    bot.infinity_polling()